# 클래스 불균형 실습

**Class Imbalance**

한 범주의 수가 훨씬 많아 정확도만으로는 성능을 판단할 수 없는 상황.

소재 분야에서 이해하기: 결함 시료가 2%뿐이면 전부 정상이라 답해도 정확도 98%가 된다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 평가 지표 문서](https://scikit-learn.org/stable/modules/model_evaluation.html)

## 1. 정확도의 함정

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

n = 4000
features = rng.normal(0, 1, (n, 4))
score = 2.0 * features[:, 0] + 1.2 * features[:, 1] - 5.0
defect = (score + rng.normal(0, 1, n) > 0).astype(int)
print('결함 비율 %.2f%% (%d / %d)' % (100 * defect.mean(), defect.sum(), n)) 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, average_precision_score

X_tr, X_te, y_tr, y_te = train_test_split(features, defect, test_size=0.3, random_state=0, stratify=defect)
for name, model in [('전부 정상이라 답하기', DummyClassifier(strategy='most_frequent')),
                    ('로지스틱 회귀', LogisticRegression(max_iter=1000)),
                    ('클래스 가중치 적용', LogisticRegression(max_iter=1000, class_weight='balanced'))]:
    model.fit(X_tr, y_tr); pred = model.predict(X_te)
    print('%-22s 정확도 %.3f  재현율 %.3f  정밀도 %.3f'
          % (name, accuracy_score(y_te, pred), recall_score(y_te, pred, zero_division=0),
             precision_score(y_te, pred, zero_division=0)))

## 2. 불균형에서는 PR 곡선이 더 정직합니다

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_auc_score

model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
probability = model.predict_proba(X_te)[:, 1]
precision, recall, _ = precision_recall_curve(y_te, probability)
plt.plot(recall, precision)
plt.axhline(y_te.mean(), color='k', ls='--', label='random baseline')
plt.xlabel('recall'); plt.ylabel('precision'); plt.legend(); plt.show()
print('ROC AUC %.3f (불균형에서도 높게 나오기 쉬움)' % roc_auc_score(y_te, probability))
print('평균 정밀도 %.3f (무작위 기준선 %.3f)' % (average_precision_score(y_te, probability), y_te.mean()))

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#class-imbalance)을 여세요.